# One Month Ahead Forecasting – Inflation Rate China

# 1: Preprocessing

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [2]:

df = pd.read_csv(
    "china_economic_raw.csv",
    sep=";",
    na_values=["#DIV/0!", "#N/A", "#VALUE!", "#REF!"]
)

df.columns = df.columns.str.strip()

# Convert Date column
df["Date"] = pd.to_datetime(df["Date"], format="%b %y")

# Sort chronologically
df = df.sort_values("Date").reset_index(drop=True)

# Define target and predictors
target = "INFL"
predictors = [
    col for col in df.columns
    if col.startswith("X") and col[1:].isdigit()
]

# Basic inspection
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(376, 167)
        Date   INFL     X1     X2     X3     X4     X5     X6     X7   X8  \
0 1995-01-01    NaN  124,1  135,7  155,6  152,6  116,8  121,9  129,8  NaN   
1 1995-02-01  -1,38  122,4  132,1  157,4  144,9  113,6  113,5  121,5  NaN   
2 1995-03-01  -0,90  121,3  130,6    155  142,7  118,7  113,6  119,2  NaN   
3 1995-04-01  -0,50  120,7    130  153,9    137  119,1  114,7  129,8  NaN   
4 1995-05-01  -0,33  120,3  129,2  150,8  132,6  119,5    116  133,2  NaN   

   ... X156 X157 X158 X159 X160 X161 X162 X163 X164 X165  
0  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[5 rows x 167 columns]
<class 'pandas.DataFrame'>
RangeIndex: 376 entries, 0 to 375
Columns: 167 entries, Date to X165
dtypes: datetime64[us](1), float64(7), str(159)
memo

In order to run the following code, convert Strings to numbers.

In [3]:
# Check which predictors are not numeric
non_numeric = df[predictors].select_dtypes(
    exclude="number"
).columns

print("Non-numeric features:")
print(non_numeric.tolist())

Non-numeric features:
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X38', 'X39', 'X40', 'X41', 'X42', 'X43', 'X45', 'X46', 'X47', 'X48', 'X49', 'X50', 'X51', 'X52', 'X53', 'X56', 'X57', 'X58', 'X59', 'X60', 'X61', 'X62', 'X63', 'X64', 'X65', 'X66', 'X67', 'X68', 'X69', 'X70', 'X71', 'X72', 'X73', 'X74', 'X75', 'X76', 'X77', 'X78', 'X79', 'X80', 'X81', 'X82', 'X83', 'X84', 'X85', 'X86', 'X87', 'X88', 'X89', 'X90', 'X91', 'X92', 'X93', 'X94', 'X95', 'X96', 'X97', 'X98', 'X99', 'X100', 'X101', 'X102', 'X103', 'X104', 'X105', 'X106', 'X107', 'X108', 'X109', 'X110', 'X111', 'X112', 'X113', 'X114', 'X115', 'X116', 'X117', 'X118', 'X119', 'X120', 'X121', 'X122', 'X123', 'X124', 'X125', 'X126', 'X127', 'X128', 'X129', 'X130', 'X131', 'X132', 'X133', 'X134', 'X135', 'X136', 'X137', 'X138', 'X139', 'X140', 'X141', 'X142'

In [4]:
# Convert predictors and target to numeric values

for col in predictors + ["INFL"]:
    original = df[col]

    converted = pd.to_numeric(
        original.astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Check if conversion introduced additional NaNs
    new_missing = original.notna() & converted.isna()

    if new_missing.any():
        print(f"{col}: Unexpected values:")
        print(original[new_missing].unique()[:10])

    df[col] = converted

# Check whether all variables are numeric
print(df[predictors].dtypes.value_counts())


Float64    165
Name: count, dtype: int64


In [5]:
missing_values = df[predictors].isna().sum()

missing_percentage = (
    df[predictors].isna().mean() * 100
)

missing_report = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percentage
})

print(
    missing_report.sort_values(
        "Missing %", ascending=False
    ).head(20)
)


      Missing Values  Missing %
X27              134  35.638298
X28              133  35.372340
X29              125  33.244681
X23              123  32.712766
X25              123  32.712766
X165             120  31.914894
X154             120  31.914894
X155             120  31.914894
X156             120  31.914894
X157             120  31.914894
X158             120  31.914894
X159             120  31.914894
X160             120  31.914894
X161             120  31.914894
X162             120  31.914894
X163             120  31.914894
X164             120  31.914894
X26              117  31.117021
X24              115  30.585106
X21              115  30.585106


High  portion of missing values indicates the necessity of further analysing nature of data. Visual inspection of dataset hints to more missing values earlier.
If this is true, which will be tested, it could be beneficial to use only a later part of the dataset, since the data quality significantly improves and by this the model would probably also significantly improve (garbage in, garbage out).

In [6]:
# Calculate percentage of missing values for each year

yearly_missing = (
    df[predictors]
    .isna()
    .groupby(df["Date"].dt.year)
    .mean()
    .mean(axis=1) * 100
)

# Display results
print(yearly_missing.round(2).to_string())

Date
1995    66.62
1996    64.24
1997    43.28
1998    40.20
1999    23.48
2000    15.81
2001    12.73
2002    12.73
2003    12.73
2004    12.12
2005     3.18
2006     1.52
2007     5.86
2008     5.56
2009     5.61
2010     5.56
2011     0.30
2012     0.56
2013     0.71
2014     1.92
2015     0.76
2016     2.32
2017     2.32
2018     2.32
2019     2.27
2020     2.27
2021     2.27
2022     2.27
2023     2.42
2024     2.47
2025     2.47
2026    11.67


From 2004 to 2005, the amount of missing values decreased significantly. For this reason, we will only use data from 2005 onwards to train and test our model. We decided to use this threshold because the completeness of the data improved a lot by almost 10 percentage points. However, we want to point out that it might also have been possible to use the data from 2000 onwards (see report). For now, we have decided to retain observations prior to 2005 for the construction of lagged features, while excluding them as separate training observations. This allows us to preserve potentially useful historical information that would otherwise be lost. Although more recent observations may be more predictive, some economic indicators might also influence inflation with longer time lags. We will therefore evaluate the relevance of different lags during model training and validation.

In [7]:
# Create a copy of our dataset
data = df.sort_values("Date").reset_index(drop=True).copy()

# Keep original values for duplicate detection later
original_values = data[predictors].copy()

# Missing values before forward fill
missing_before = data[predictors].isna().sum().sum()

# Forward fill, maximum 2 consecutive months
data[predictors] = data[predictors].ffill(limit=2)

# Missing values after forward fill
missing_after = data[predictors].isna().sum().sum()

print("Missing values before:", missing_before)
print("Missing values after:", missing_after)


Missing values before: 7183
Missing values after: 6053


We decided to use forward fill to replace short gaps of up to two consecutive months with the last available value. Since forward fill only uses previous observations, we do not use any future data to fill missing values. We limit it to two months because using older observations for longer gaps might be less reliable. Longer gaps will remain as NaN for now.

We also want to remove duplicate features since they do not provide additional information to our models. Before removing them, we check if the variables have identical values wherever both have observations. If they do, we first combine them so that we do not lose any data if one column has missing values that are available in the other column. We then remove one of the duplicate columns.

In [8]:
# Create a fresh copy of the dataset
data = df.sort_values("Date").reset_index(drop=True).copy()

# Convert all predictors to numeric
for col in predictors:
    original = data[col]

    converted = pd.to_numeric(
        original.astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Check whether conversion creates new missing values
    new_missing = original.notna() & converted.isna()

    if new_missing.any():
        raise ValueError(
            f"Non-numeric values found in {col}: "
            f"{original[new_missing].unique()[:10]}"
        )

    data[col] = converted.astype(float)

# Keep original numeric values for duplicate detection
original_values = data[predictors].copy()

# Forward fill short gaps
data[predictors] = data[predictors].ffill(limit=2)

# Detect and merge duplicate features
features = predictors.copy()
merged_pairs = []

for i, keep in enumerate(predictors):
    if keep not in features:
        continue

    for remove in predictors[i + 1:]:
        if remove not in features:
            continue

        a = original_values[keep]
        b = original_values[remove]

        # Compare only observations where both have values
        overlap = a.notna() & b.notna()

        if overlap.sum() < 24:
            continue

        if a[overlap].nunique() < 3:
            continue

        # Check whether values match
        if np.allclose(
            a[overlap].to_numpy(dtype=float),
            b[overlap].to_numpy(dtype=float),
            rtol=1e-9,
            atol=1e-8
        ):

            # Combine available observations
            merged = a.combine_first(b)

            original_values[keep] = merged
            data[keep] = merged.ffill(limit=2)

            # Remove redundant column
            data = data.drop(columns=remove)
            features.remove(remove)

            merged_pairs.append((keep, remove))

print("Merged duplicate columns:", merged_pairs)
print("Remaining features:", len(features))


Merged duplicate columns: [('X123', 'X129'), ('X124', 'X130'), ('X125', 'X131'), ('X126', 'X132')]
Remaining features: 161


For the initial feature selection, we focus on missing values between 2005 and 2020 rather than over the entire dataset. Since we plan to use observations from 2005 onwards for our predictions, we do not want to remove features just because they had many missing values in the 1990s. These earlier values might still be useful for lags, but the completeness of the features in our main observation period is more relevant.

For now, we use a threshold of 20% missing values. We also remove constant features since they do not provide any information about changes in our target variable.

In [9]:

# Reference period for initial feature filtering
reference_mask = data["Date"].between(
    "2005-01-01", "2023-12-01"
)

# Missing values per feature
missing_ratio = (
    data.loc[reference_mask, features]
    .isna()
    .mean()
)

# Display highest missing ratios
print(
    (missing_ratio * 100)
    .sort_values(ascending=False)
    .head(20)
)

# Remove features with more than 20% missing
threshold = 0.20

drop_missing = missing_ratio[
    missing_ratio > threshold
].index.tolist()

# Identify constant features
drop_constant = [
    col for col in features
    if data.loc[
        reference_mask, col
    ].nunique(dropna=True) <= 1
]

# Combine exclusions
drop_features = list(set(
    drop_missing + drop_constant
))

# Remove features
data = data.drop(columns=drop_features)

features = [
    col for col in features
    if col not in drop_features
]

print("Dropped for missing values:", drop_missing)
print("Dropped constant features:", drop_constant)
print("Remaining features:", len(features))

X27     5.263158
X28     5.263158
X29     1.754386
X23     1.315789
X25     1.315789
X123    0.000000
X107    0.000000
X108    0.000000
X109    0.000000
X110    0.000000
X111    0.000000
X112    0.000000
X113    0.000000
X114    0.000000
X115    0.000000
X116    0.000000
X117    0.000000
X118    0.000000
X119    0.000000
X120    0.000000
dtype: float64
Dropped for missing values: []
Dropped constant features: []
Remaining features: 161


No variable was removed because of missing values for now. Since 6.25% is the highest portion of missing values in all of the given features we can keep all of them for a start.

Lastly, we want to check whether some features directly explain our target variable. If a feature contains the same information as INFL, using it could lead to data leakage and unrealistically good predictions. We therefore check whether any of our features have an exact linear relationship with INFL and remove them if necessary.

In [10]:

target = "INFL"

leakage_features = []

for col in features:

    # Use observations where both values exist
    pair = data.loc[
        reference_mask, [col, target]
    ].dropna()

    if len(pair) < 24 or pair[col].nunique() <= 1:
        continue

    x = pair[col].to_numpy()
    y = pair[target].to_numpy()

    # Test whether INFL = a * X + b
    A = np.column_stack([x, np.ones_like(x)])

    slope, intercept = np.linalg.lstsq(
        A, y, rcond=None
    )[0]

    predicted = slope * x + intercept

    max_error = np.max(np.abs(y - predicted))

    if max_error < 1e-8:
        leakage_features.append(col)

print("Directly explaining features:", leakage_features)

Directly explaining features: []


None of the individual features were found to have an exact linear relationship with the target variable. However, this does not rule out other forms of data leakage. To avoid using information from the same month as our prediction target, we will only use lagged predictor variables.

For prediction, we use lagged features. We move all features by one month.

In [11]:
# Shift all predictors by one month
X = data[features].shift(1).copy()

X.columns = [
    col + "_lag1" for col in features
]

We will add some potentially useful predictors to the already existing ones. For one, previous inflation rates might be a useful predictor of this month's inflation. We use last month's inflation and inflation from 3 and 12 months ago. We refrained from any other engineered features since, in relation to our number of observations, we already have a large number of features. Moreover, we plan on using XGBoost and decision trees, which can learn non-linear relationships from the input data.

In [12]:
# Historical inflation
X["INFL_lag1"] = data["INFL"].shift(1)
X["INFL_lag3"] = data["INFL"].shift(3)
X["INFL_lag12"] = data["INFL"].shift(12)

# Average inflation over the previous 3 months
X["INFL_mean3"] = (
    data["INFL"].shift(1).rolling(3).mean()
)

Because seasonality could be a relevant factor in inflation, we decode the month of the observation.

In [13]:
month = data["Date"].dt.month

X["month_sin"] = np.sin(2 * np.pi * month / 12)
X["month_cos"] = np.cos(2 * np.pi * month / 12)

In [15]:
# Combine target and engineered features
prepared = pd.concat(
    [data[["Date", "INFL"]], X],
    axis=1
)

# Keep only observations from 2005 onward
prepared = prepared[
    (prepared["Date"] >= "2005-01-01") &
    prepared["INFL"].notna()
].copy()

# Training observations only
outlier_train = prepared[
    prepared["Date"] < "2024-01-01"
].copy()

# Exclude binary and seasonal features from IQR audit
outlier_cols = [
    col for col in X.columns
    if col not in [
        "PMI_expansion_lag1",
        "month_sin",
        "month_cos"
    ]
]

Q1 = outlier_train[outlier_cols].quantile(0.25)
Q3 = outlier_train[outlier_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = (
    (outlier_train[outlier_cols] < Q1 - 1.5 * IQR) |
    (outlier_train[outlier_cols] > Q3 + 1.5 * IQR)
)

print(outliers.sum().sort_values(ascending=False).head(20))


X1_lag1      29
X165_lag1    29
X9_lag1      27
X18_lag1     27
X53_lag1     26
X163_lag1    22
X157_lag1    21
X16_lag1     20
X160_lag1    20
X17_lag1     20
X4_lag1      19
X155_lag1    19
X19_lag1     18
X83_lag1     17
X143_lag1    16
X2_lag1      16
X146_lag1    15
X81_lag1     14
X82_lag1     14
X85_lag1     14
dtype: Int64


We used the IQR method to identify potential outliers in our predictor variables. However, we decided not to remove them for now. Since we are working with economic data, extreme values might reflect actual economic events rather than errors in the dataset. Removing these observations could therefore lead to a loss of relevant information for predicting inflation. If necessary, we might reconsider how to handle outliers depending on the performance of our models.

Up to this point, we have not replaced all missing values.

In [16]:
# Analyze remaining missing values after feature engineering

missing_values = prepared[X.columns].isna().sum()

missing_percentage = (
    prepared[X.columns].isna().mean() * 100
)

missing_report = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percentage
})

print(
    missing_report[
        missing_report["Missing Values"] > 0
    ].sort_values(
        "Missing %", ascending=False
    )
)

print(
    "Total remaining missing values:",
    missing_values.sum()
)


           Missing Values  Missing %
X27_lag1               13   5.078125
X28_lag1               13   5.078125
X29_lag1                5   1.953125
X23_lag1                4   1.562500
X25_lag1                4   1.562500
X156_lag1               1   0.390625
X164_lag1               1   0.390625
X163_lag1               1   0.390625
X162_lag1               1   0.390625
X161_lag1               1   0.390625
X160_lag1               1   0.390625
X159_lag1               1   0.390625
X158_lag1               1   0.390625
X157_lag1               1   0.390625
X154_lag1               1   0.390625
X155_lag1               1   0.390625
X115_lag1               1   0.390625
X114_lag1               1   0.390625
X113_lag1               1   0.390625
X107_lag1               1   0.390625
X106_lag1               1   0.390625
X105_lag1               1   0.390625
X74_lag1                1   0.390625
X165_lag1               1   0.390625
Total remaining missing values: 58


We will handle these missing values with methods like median imputation during the training of the models to avoid data leakage.

The next step is, do define a uniform test and train split. After that, the models can be trained and afterwards tested on that data. Further data processing techniques that are fitted to the different models will be used in the respective pipelines.

In [17]:
# Chronological train-test split

# Training: January 2005 - December 2023
train = prepared[
    (prepared["Date"] >= "2005-01-01") &
    (prepared["Date"] < "2024-01-01")
].copy()

# Testing: January 2024 - April 2026
test = prepared[
    (prepared["Date"] >= "2024-01-01") &
    (prepared["Date"] <= "2026-04-01")
].copy()

# Separate predictors and target
X_train = train.drop(columns=["Date", "INFL"])
y_train = train["INFL"]

X_test = test.drop(columns=["Date", "INFL"])
y_test = test["INFL"]

# Check number of observations
print("Training observations:", len(train))
print("Testing observations:", len(test))

Training observations: 228
Testing observations: 28


We decided to use the time period from 2005 to 2023 for training purposes, which contains a total of 228 observations. For testing, we use the data from January 2024 to April 2026 (28 data points).

We decided not to use a separate validation set, as we will use time-series cross-validation for hyperparameter tuning within our training data. This also allows us to use more observations for training, which might be beneficial given the relatively large number of predictor variables.

We chose this split because we want to keep as many observations as possible for training while still having a sufficiently long period of unseen data to evaluate our models. The test period covers 28 consecutive months, allowing us to evaluate how well our models predict inflation on more recent data.

Since we are working with time-series data, we also need to make sure that the training data always comes before the testing data to avoid data leakage.